# Notebook 04 — Fannie Mae Florida restriction (positional filter)

**Input**: `2017Q1.csv` (8.4 GB, 113 columns per Notebook 01 result).

**Output**: `fannie_mae_2017Q1_FL.parquet` — Florida-only subset.

**Why positional not name-based**: your file has 113 columns but Fannie Mae's
originally-documented layout has 108. Fannie Mae has added 5 new fields
(likely delinquency resolution / payment deferral / non-interest bearing UPB
fields introduced post-2020). Rather than guess where they were inserted, this
notebook:

1. Reads a small sample without column names
2. **Auto-detects the STATE column** by finding a column whose values look like
   US state abbreviations (`FL`, `CA`, `TX`, etc.)
3. Verifies key columns (LOAN_ID, ZIP, DTI, CSCORE_B, ACT_PERIOD) by content
4. Filters the full file by STATE position, saves with proper column names
   for the first 108 columns and `col_108` … `col_112` for the extra 5

You can rename `col_108` – `col_112` later once you identify what they are
from the Fannie Mae updated glossary. They are unlikely to affect the baseline
modelling since they are new post-origination fields (payment deferral,
delinquency resolution) that will be null for most 2017-vintage loans until
event dates.

Runtime: 5–15 minutes for the chunked read.

## Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# TODO: adjust DATA_DIR to your data folder
DATA_DIR = Path("your/data/path/here")

FANNIE_RAW    = DATA_DIR / "2017Q1.csv"
OUT_FANNIE_FL = DATA_DIR / "fannie_mae_2017Q1_FL.parquet"

FANNIE_SEP        = '|'    # confirmed from Notebook 01
FANNIE_HAS_HEADER = False  # confirmed from Notebook 01
N_COLS_EXPECTED   = 113    # confirmed from Notebook 01

## Diagnostic — sample first rows to identify columns

Reads the first 500 rows *without column names* and shows sample values for
each of the 113 columns. This lets us find where STATE lives.

In [ ]:
# Read a small diagnostic sample WITHOUT column names
sample = pd.read_csv(
    FANNIE_RAW,
    sep=FANNIE_SEP,
    header=None,
    nrows=500,
    dtype=str,          # everything as string for diagnostic
    low_memory=False,
)

print(f"Diagnostic sample shape: {sample.shape}")
assert sample.shape[1] == N_COLS_EXPECTED, (
    f"Expected {N_COLS_EXPECTED} columns, got {sample.shape[1]}"
)

# Show first 3 non-null unique values per column
print("\nColumn-by-column preview (first 3 non-null unique values):")
for i in range(sample.shape[1]):
    col_vals = sample.iloc[:, i].dropna().astype(str).unique()[:3]
    preview = " | ".join(col_vals)[:70]
    print(f"  col_{i:3d}: {preview}")

## Auto-detect STATE column

Search columns for values that look like US state abbreviations. STATE should
be a column where most values are exactly 2 uppercase letters from a fixed
set of US states.

In [ ]:
US_STATES = set([
    'AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA','HI','ID','IL','IN','IA',
    'KS','KY','LA','ME','MD','MA','MI','MN','MS','MO','MT','NE','NV','NH','NJ',
    'NM','NY','NC','ND','OH','OK','OR','PA','RI','SC','SD','TN','TX','UT','VT',
    'VA','WA','WV','WI','WY','DC','PR','VI','GU'
])

state_col_candidates = []
for i in range(sample.shape[1]):
    vals = sample.iloc[:, i].dropna().astype(str)
    if len(vals) == 0:
        continue
    # Fraction of values that are US state codes
    n_state = vals.isin(US_STATES).sum()
    frac = n_state / len(vals)
    if frac > 0.90:  # column is dominantly US state codes
        state_col_candidates.append((i, frac, vals.iloc[0]))

print("STATE column candidates (>90% US state codes):")
for i, frac, first_val in state_col_candidates:
    print(f"  col_{i}: {frac:.1%} state-code, sample value = '{first_val}'")

assert len(state_col_candidates) == 1, (
    f"Expected exactly 1 STATE column, found {len(state_col_candidates)}. "
    "Inspect the diagnostic preview above."
)

STATE_COL = state_col_candidates[0][0]
print(f"\n>>> STATE column position: col_{STATE_COL}")

# Sanity: how many FL rows in the 500-row sample?
n_fl_sample = (sample.iloc[:, STATE_COL] == 'FL').sum()
print(f">>> Florida rows in 500-row sample: {n_fl_sample}")

## Verify other key columns by content

Confirm we can find LOAN_ID (long integer string), ZIP (3-digit), DTI (1–65
range), CSCORE_B (300–850 range), ACT_PERIOD (YYYYMM format).

In [ ]:
def find_column(pattern_fn, name):
    matches = []
    for i in range(sample.shape[1]):
        vals = sample.iloc[:, i].dropna().astype(str)
        if len(vals) < 10:
            continue
        try:
            if pattern_fn(vals):
                matches.append(i)
        except Exception:
            pass
    print(f"{name} candidates: {matches}")
    return matches

# LOAN_ID: 12-digit-ish integer strings, all unique
find_column(
    lambda v: v.str.len().between(10, 14).mean() > 0.9 and v.nunique() / len(v) > 0.5,
    "LOAN_ID"
)

# ACT_PERIOD: YYYYMM format, mostly starts with 20
find_column(
    lambda v: v.str.match(r"^20\d{4}$").mean() > 0.9,
    "ACT_PERIOD (YYYYMM)"
)

# ZIP: 3-digit zip
find_column(
    lambda v: v.str.match(r"^\d{3}$").mean() > 0.8,
    "ZIP (3-digit)"
)

# CSCORE_B: 3-digit int in 300-850
def is_credit_score(v):
    numeric = pd.to_numeric(v, errors='coerce').dropna()
    if len(numeric) == 0:
        return False
    return numeric.between(300, 850).mean() > 0.9

find_column(is_credit_score, "CSCORE_B (300-850)")

# DTI: numeric, 1-65 typical range
def is_dti(v):
    numeric = pd.to_numeric(v, errors='coerce').dropna()
    if len(numeric) < 50:
        return False
    return numeric.between(1, 65).mean() > 0.85

find_column(is_dti, "DTI (1-65)")

## Column-name assignment

We use the standard 108-column Fannie Mae layout for positions 0–107 and
placeholder names `col_108` – `col_112` for the 5 new fields. If the
diagnostic above shows STATE at position 30 as expected in the standard
layout, we can trust that positions 0–107 match the standard names.

In [ ]:
FANNIE_COLUMNS_108 = [
    'POOL_ID', 'LOAN_ID', 'ACT_PERIOD', 'CHANNEL', 'SELLER', 'SERVICER',
    'MASTER_SERVICER', 'ORIG_RATE', 'CURR_RATE', 'ORIG_UPB', 'ISSUANCE_UPB',
    'CURRENT_UPB', 'ORIG_TERM', 'ORIG_DATE', 'FIRST_PAY', 'LOAN_AGE',
    'REM_MONTHS', 'ADJ_REM_MONTHS', 'MATR_DT', 'ORIG_LTV', 'ORIG_CLTV',
    'NUM_BORR', 'DTI', 'CSCORE_B', 'CSCORE_C', 'FIRST_FLAG', 'PURPOSE',
    'PROP', 'NO_UNITS', 'OCC_STAT', 'STATE', 'MSA', 'ZIP', 'MI_PCT',
    'PRODUCT', 'PPMT_FLG', 'IO', 'FIRST_PAY_IO', 'MNTHS_TO_AMTZ_IO',
    'DLQ_STATUS', 'PMT_HISTORY', 'MOD_FLAG', 'MI_CANCEL_FLAG', 'ZERO_BAL_CODE',
    'ZB_DTE', 'LAST_UPB', 'RPRCH_DTE', 'CURR_SCHD_PRNCPL', 'TOT_SCHD_PRNCPL',
    'UNSCHD_PRNCPL_CURR', 'LAST_PAID_INSTALLMENT_DATE', 'FORECLOSURE_DATE',
    'DISPOSITION_DATE', 'FORECLOSURE_COSTS',
    'PROPERTY_PRESERVATION_AND_REPAIR_COSTS',
    'ASSET_RECOVERY_COSTS', 'MISCELLANEOUS_HOLDING_EXPENSES_AND_CREDITS',
    'ASSOCIATED_TAXES_FOR_HOLDING_PROPERTY', 'NET_SALES_PROCEEDS',
    'CREDIT_ENHANCEMENT_PROCEEDS', 'REPURCHASES_MAKE_WHOLE_PROCEEDS',
    'OTHER_FORECLOSURE_PROCEEDS', 'NON_INTEREST_BEARING_UPB',
    'PRINCIPAL_FORGIVENESS_AMOUNT', 'ORIGINAL_LIST_START_DATE',
    'ORIGINAL_LIST_PRICE', 'CURRENT_LIST_START_DATE', 'CURRENT_LIST_PRICE',
    'ISSUE_SCOREB', 'ISSUE_SCOREC', 'CURR_SCOREB', 'CURR_SCOREC',
    'MI_TYPE', 'SERV_IND', 'CURRENT_PERIOD_MODIFICATION_LOSS_AMOUNT',
    'CUMULATIVE_MODIFICATION_LOSS_AMOUNT',
    'CURRENT_PERIOD_CREDIT_EVENT_NET_GAIN_OR_LOSS',
    'CUMULATIVE_CREDIT_EVENT_NET_GAIN_OR_LOSS',
    'HOMEREADY_PROGRAM_INDICATOR',
    'FORECLOSURE_PRINCIPAL_WRITE_OFF_AMOUNT',
    'RELOCATION_MORTGAGE_INDICATOR',
    'ZERO_BALANCE_CODE_CHANGE_DATE', 'LOAN_HOLDBACK_INDICATOR',
    'LOAN_HOLDBACK_EFFECTIVE_DATE', 'DELINQUENT_ACCRUED_INTEREST',
    'PROPERTY_INSPECTION_WAIVER_INDICATOR',
    'HIGH_BALANCE_LOAN_INDICATOR',
    'ARM_5_YR_INDICATOR', 'ARM_PRODUCT_TYPE',
    'MONTHS_UNTIL_FIRST_PAYMENT_RESET',
    'MONTHS_BETWEEN_SUBSEQUENT_PAYMENT_RESET',
    'INTEREST_RATE_CHANGE_DATE',
    'PAYMENT_CHANGE_DATE', 'ARM_INDEX', 'ARM_CAP_STRUCTURE',
    'INITIAL_INTEREST_RATE_CAP', 'PERIODIC_INTEREST_RATE_CAP',
    'LIFETIME_INTEREST_RATE_CAP', 'MARGIN', 'BALLOON_INDICATOR',
    'PLAN_NUMBER', 'FORBEARANCE_INDICATOR',
    'HIGH_LOAN_TO_VALUE_REFINANCE_OPTION_INDICATOR',
    'DEAL_NAME', 'RE_PROCS_FLAG', 'ADR_TYPE', 'ADR_COUNT', 'ADR_UPB'
]

# Extra 5 fields (post-2020 additions) — placeholder names
EXTRA_COLUMNS = [f'col_{i}' for i in range(108, 113)]

FANNIE_COLUMNS = FANNIE_COLUMNS_108 + EXTRA_COLUMNS
assert len(FANNIE_COLUMNS) == N_COLS_EXPECTED

# Trust the standard layout only if STATE detected at position 30
STATE_STANDARD_POS = 30
if STATE_COL == STATE_STANDARD_POS:
    print(f"✓ STATE detected at position {STATE_COL} — matches standard 108-column layout.")
    print(f"  Assigning names positions 0-107 from standard layout, 108-112 as col_108-col_112.")
else:
    print(f"⚠ STATE detected at position {STATE_COL}, NOT the standard position 30.")
    print(f"  The extra 5 columns may have been inserted somewhere in the middle,")
    print(f"  not at the end. Column names will be misaligned.")
    print(f"  DO NOT PROCEED — inspect the diagnostic preview and share with Claude.")
    raise SystemExit("STATE at unexpected position; halting.")

## Chunked read + Florida filter

Now we know STATE is at position 30 and we can trust the standard layout for
positions 0–107. Read chunks with proper column names, filter by STATE == 'FL',
concatenate and save.

In [ ]:
CHUNK_SIZE = 500_000

read_kwargs = dict(
    sep=FANNIE_SEP,
    chunksize=CHUNK_SIZE,
    header=None,
    names=FANNIE_COLUMNS,
    dtype={
        'LOAN_ID': str, 'POOL_ID': str, 'ACT_PERIOD': str,
        'STATE': str, 'ZIP': str, 'MSA': str,
    },
    low_memory=False,
)

fl_chunks   = []
total_rows  = 0
fl_rows     = 0

print("Starting chunked read ...")
for i, chunk in enumerate(pd.read_csv(FANNIE_RAW, **read_kwargs)):
    fl_chunk = chunk[chunk['STATE'] == 'FL']
    fl_chunks.append(fl_chunk)
    total_rows += len(chunk)
    fl_rows    += len(fl_chunk)
    if i % 10 == 0:
        print(f"  Chunk {i}: cumulative rows read = {total_rows:,}, FL kept = {fl_rows:,}")

print(f"\nConcatenating {len(fl_chunks)} chunks ...")
fl_data = pd.concat(fl_chunks, ignore_index=True)
del fl_chunks

print(f"Total rows read: {total_rows:,}")
print(f"Florida rows kept: {fl_rows:,}")
print(f"Unique FL loans: {fl_data['LOAN_ID'].nunique():,}")
print(f"ACT_PERIOD range: {fl_data['ACT_PERIOD'].min()} to {fl_data['ACT_PERIOD'].max()}")

## Zero-pad ZIP3 and save

In [ ]:
# Fannie Mae reports ZIP as 3-digit; ensure it stays 3-digit with leading zeros
fl_data['ZIP'] = (
    fl_data['ZIP']
    .astype(str)
    .str.replace(r'\.0$', '', regex=True)
    .str.zfill(3)
)

fl_data.to_parquet(OUT_FANNIE_FL, index=False)
print(f"Saved: {OUT_FANNIE_FL}  ({OUT_FANNIE_FL.stat().st_size / 1e6:.1f} MB)")

## Next step

The Florida parquet is ready for Notebook 05 (summary) and downstream Chapter 5
baseline work.

If you later need to identify what `col_108` – `col_112` actually are, the
Fannie Mae glossary at https://capitalmarkets.fanniemae.com/media/6931/display
lists the current field layout. They are almost certainly not needed for the
baseline modelling — most are post-origination fields (payment deferral,
delinquency resolution) that will be null or empty for most 2017 loans.